# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [2]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
)

CONFIG = Configuration(
)

# Data

### Rotation 'r'
- N: north, AKA "0"
- E: east, AKA "R"
- S: south, AKA "2"
- W: west, AKA "L"

### Playfield
- N: Empty
- G: Garbage

### Won and ranking
In case we want to use only won games. Max ranking25000

'game_id', 'subframe', 'won', 'playfield', 'x', 'y', 'r', 'placed',
       'hold', 'next', 'cleared', 'garbage_cleared', 'attack', 't_spin', 'btb',
       'combo', 'immediate_garbage', 'incoming_garbage', 'rating', 'glicko',
       'glicko_rd'

In [41]:
raw_df = pd.read_csv(CONFIG.raw_dataset_path)
print(raw_df.columns)

raw_df = raw_df.sort_values(by=['game_id', 'subframe'])

Index(['game_id', 'subframe', 'won', 'playfield', 'x', 'y', 'r', 'placed',
       'hold', 'next', 'cleared', 'garbage_cleared', 'attack', 't_spin', 'btb',
       'combo', 'immediate_garbage', 'incoming_garbage', 'rating', 'glicko',
       'glicko_rd'],
      dtype='str')


In [ ]:
raw_df.head(10)


# if no hold -> first on the queue + remove first from the queue
# if hold has changed -> current = hold, hold = next from the queue if empty
# if hold has changed -> current = hold, hold = placed

,game_id,subframe,won,playfield,x,y,r,placed,hold,next,cleared,garbage_cleared,attack,t_spin,btb,combo,immediate_garbage,incoming_garbage,rating,glicko,glicko_rd
0,1,67,1,NaN,4,0,N,I,N,JZSOTLSLIOJZTJ,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
1,1,170,1,NNNIIII,4,1,N,Z,J,SOTLSLIOJZTJTZ,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
2,1,339,1,NNNIIIINNNNNNNZZNNNNNNNZZ,6,1,E,S,J,OTLSLIOJZTJTZS,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
3,1,472,1,NNNIIIISNNNNNNZZSSNNNNNZZNS,8,0,N,O,J,TLSLIOJZTJTZSO,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
4,1,602,1,NNNIIIISOONNNNZZSSOONNNZZNS,0,1,E,J,T,LSLIOJZTJTZSOI,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
5,1,716,1,JJNIIIISOOJNNNZZSSOOJNNZZNS,8,2,N,L,T,SLIOJZTJTZSOIL,0,0,0,N,0,0,0,0,24748.521484,2701.887695,62.212624
6,1,952,1,JJNIIIISOOJNNNZZSSOOJNNZZNSLLLNNNNNNNL,2,1,S,T,S,LIOJZTJTZSOILS,2,0,4,F,0,0,0,0,24748.521484,2701.887695,62.212624
7,1,1093,1,JNNZZNSLLLNNNNNNNL,9,2,W,L,S,IOJZTJTZSOILSZ,0,0,0,N,1,1,0,0,24748.521484,2701.887695,62.212624
8,1,1411,1,JNNZZNSLLLNNNNNNNLLLNNNNNNNNNLNNNNNNNNNL,2,0,N,S,I,OJZTJTZSOILSZJ,0,0,0,N,1,0,0,0,24748.521484,2701.887695,62.212624
9,1,1586,1,JSSZZNSLLLNNSSNNNLLLNNNNNNNNNLNNNNNNNNNL,0,1,N,O,I,JZTJTZSOILSZJL,0,0,0,N,1,0,0,0,24748.521484,2701.887695,62.212624


In [45]:
print('- Loading raw dataset...')
raw_df = pd.read_csv(CONFIG.raw_dataset_path)
print(raw_df.columns)

print('- Sorting and cleaning dataset...')
raw_df = raw_df.sort_values(by=['game_id', 'subframe'])
raw_df['subframe'] = raw_df.groupby('game_id').cumcount()
raw_df['playfield'] = raw_df['playfield'].fillna('N')
raw_df['playfield_next'] = raw_df['playfield'].shift(-1).fillna('N')

print('- Dropping duplicate game_id rows...')
mask_keep_all_but_last = raw_df.duplicated(subset=['game_id'], keep='last')
raw_df = raw_df[mask_keep_all_but_last].copy()


print('- Real current to prevent always placing the current one rather than also the hold...')

# Fix: data pipeline systematically swaps J and L piece labels
# (coordinate mirror in the TETR.IO extractor)
# swap_jl = {'J': 'L', 'L': 'J'}
# raw_df['placed'] = raw_df['placed'].replace(swap_jl)
# raw_df['hold'] = raw_df['hold'].replace(swap_jl)

# if no hold -> first on the queue + remove first from the queue
# if hold has changed and was empty -> current = hold, hold = current
# 1. Get the previous 'hold' value for each game (defaults to 'N' for the first row)
prev_hold = raw_df.groupby('game_id')['hold'].shift(1).fillna('N')
# 2. Check if the hold value changed compared to the previous frame
hold_changed = prev_hold != raw_df['hold']
# 3. Apply your logic cleanly using np.where(condition, value_if_true, value_if_false)
raw_df['real_current'] = np.where(hold_changed, raw_df['hold'], raw_df['placed'])
raw_df['real_hold'] = np.where(hold_changed, raw_df['placed'], raw_df['hold'])
mask = raw_df['real_hold'] == 'N'
raw_df['real_hold'] = np.where(mask, raw_df['next'].str[0], raw_df['real_hold'])
raw_df['next'] = np.where(mask, raw_df['next'].str[1:], raw_df['next'])


print('- Processing garbage columns...')
# If lines were cleared, no garbage
# raw_df['immediate_garbage'] = np.where(raw_df['cleared'] > 0, 0, raw_df['immediate_garbage'])
# Cap at 8, as that's the max garbage that can be received in one frame
raw_df['immediate_garbage'] = raw_df['immediate_garbage'].clip(upper=8)

print('- Saving processed dataset...')
# raw_df = raw_df.groupby('game_id')
columns = ['game_id', 'subframe', 'playfield', 'playfield_next', 'placed', 'hold', 'real_current', 'real_hold', 'next','won', 'rating', 'immediate_garbage', 'incoming_garbage', 'btb', 'combo']
raw_df[columns].to_csv(CONFIG.processed_dataset_path, index=False)

- Loading raw dataset...
Index(['game_id', 'subframe', 'won', 'playfield', 'x', 'y', 'r', 'placed',
       'hold', 'next', 'cleared', 'garbage_cleared', 'attack', 't_spin', 'btb',
       'combo', 'immediate_garbage', 'incoming_garbage', 'rating', 'glicko',
       'glicko_rd'],
      dtype='str')
- Sorting and cleaning dataset...
- Dropping duplicate game_id rows...
- Real current to prevent always placing the current one rather than also the hold...
- Processing garbage columns...
- Saving processed dataset...


In [35]:
N = len(raw_df)
count = raw_df['immediate_garbage'].value_counts()
raw_probs = count / N
raw_probs

immediate_garbage
0    0.869814
1    0.031768
2    0.019234
4    0.019229
5    0.019044
3    0.012862
6    0.012580
8    0.010657
7    0.004812
Name: count, dtype: float64

In [36]:
real_garbage = raw_df[raw_df['immediate_garbage'] > 0]
count = real_garbage['immediate_garbage'].value_counts()
garbage_probs = count / len(real_garbage)
garbage_probs

immediate_garbage
1    0.244018
2    0.147745
4    0.147703
5    0.146281
3    0.098800
6    0.096634
8    0.081859
7    0.036961
Name: count, dtype: float64

In [37]:
print(f'{1 - 0.869814:.4f}') 

0.1302


In [51]:
raw_df[raw_df['game_id'] == 20][columns].head(10)


,game_id,subframe,playfield,playfield_next,placed,hold,real_current,real_hold,next,won,rating,immediate_garbage,incoming_garbage,btb,combo
1847,20,0,N,NNNNJJJNNNNNNNNNJ,J,Z,Z,J,TLSOIOJILTZSLJ,0,24748.521484,0,0,0,0
1848,20,1,NNNNJJJNNNNNNNNNJ,NNNNJJJNNNNNNNNZJNNNNNNNNZZNNNNNNNNNZ,Z,T,T,Z,LSOIOJILTZSLJI,0,24748.521484,0,0,0,0
1849,20,2,NNNNJJJNNNNNNNNZJNNNNNNNNZZNNNNNNNNNZ,LLLNJJJNNNLNNNNZJNNNNNNNNZZNNNNNNNNNZ,L,T,L,T,SOIOJILTZSLJIZ,0,24748.521484,0,0,0,0
1850,20,3,LLLNJJJNNNLNNNNZJNNNNNNNNZZNNNNNNNNNZ,LLLNJJJNNNLSNNNZJNNNSSNNNZZNNNSNNNNNZ,S,T,S,T,OIOJILTZSLJIZO,0,24748.521484,0,0,0,0
1851,20,4,LLLNJJJNNNLSNNNZJNNNSSNNNZZNNNSNNNNNZ,LLLNJJJOONLSNNNZJOONSSNNNZZNNNSNNNNNZ,O,T,O,T,IOJILTZSLJIZOT,0,24748.521484,0,0,0,0
1852,20,5,LLLNJJJOONLSNNNZJOONSSNNNZZNNNSNNNNNZ,LLLNJJJOOILSNNNZJOOISSNNNZZNNISNNNNNZNNI,I,T,I,T,OJILTZSLJIZOTS,0,24748.521484,0,0,0,0
1853,20,6,LLLNJJJOOILSNNNZJOOISSNNNZZNNISNNNNNZNNI,LLLNJJJOOILSNNNZJOOISSNNNZZOOISNNNNNZOOI,O,T,O,T,JILTZSLJIZOTSI,0,24748.521484,0,0,0,0
1854,20,7,LLLNJJJOOILSNNNZJOOISSNNNZZOOISNNNNNZOOI,LLLNJJJOOILSJNNZJOOISSJNNZZOOISJJNNNZOOI,J,T,J,T,ILTZSLJIZOTSIT,0,24748.521484,0,0,0,0
1855,20,8,LLLNJJJOOILSJNNZJOOISSJNNZZOOISJJNNNZOOI,LSJINZJOOISSJINZZOOISJJINNZOOI,I,T,I,T,LTZSLJIZOTSITO,0,24748.521484,0,0,0,0
1856,20,9,LSJINZJOOISSJINZZOOISJJINNZOOI,N,L,T,L,T,TZSLJIZOTSITOS,0,24748.521484,0,0,0,1


In [45]:
i = 9

In [46]:
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['playfield'].values[0])
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['playfield_next'].values[0])
print(raw_df[(raw_df['game_id'] == 18) & (raw_df['subframe'] == i)]['immediate_garbage'].values[0])

i += 1

GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIIIITNLJLZZNSSNNNNLLZNNSS
GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIIIITNLJLZZZSSNNNNLLZZZSSNNNNNNNZ
0


In [47]:
raw_df[(raw_df['game_id'] == 18) & ((raw_df['subframe'] == 10) | (raw_df['subframe'] == 9) | (raw_df['subframe'] == 11))][columns]

,game_id,subframe,playfield,playfield_next,real_current,placed,real_hold,hold,next,won,rating,immediate_garbage,incoming_garbage,btb,combo
1671,18,9,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,Z,Z,S,S,TIOSZOTJLIJSTZ,0,24748.521484,0,0,0,0
1672,18,10,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLJLZIII...,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLNLLZZZ...,T,T,S,S,IOSZOTJLIJSTZL,0,24748.521484,0,5,0,0
1673,18,11,GGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOONNTTLNLLZZZ...,GGNGGGGGGGGGNGGGGGGGGGNGGGGGGGJJJOONNTLLJJJOON...,I,I,S,S,OSZOTJLIJSTZLI,0,24748.521484,1,1,1,1


### Split

In [48]:
print('Loading processed dataset...')
processed_df = pd.read_csv(CONFIG.processed_dataset_path)

print('Other stuff...')

# Shuffle by game_id to prevent data leakage (keeping frames of the same game together)
unique_games = pd.Series(processed_df['game_id'].unique()).sample(frac=1, random_state=CONFIG.seed)

# Calculate split indices assuming CONFIG sizes are floats (e.g., 0.1 for 10%)
train_end = int(len(unique_games) * (1 - CONFIG.test_size - CONFIG.val_size))
val_end = train_end + int(len(unique_games) * CONFIG.val_size)

# Partition the shuffled game IDs
train_ids = unique_games.iloc[:train_end]
val_ids = unique_games.iloc[train_end:val_end]
test_ids = unique_games.iloc[val_end:]

# Create the final partitions
train_df = processed_df[processed_df['game_id'].isin(train_ids)].copy()
train_df = train_df.sample(frac=1, random_state=CONFIG.seed).reset_index(drop=True)
val_df = processed_df[processed_df['game_id'].isin(val_ids)].copy()
test_df = processed_df[processed_df['game_id'].isin(test_ids)].copy()

print('Saving partitions...')
print(f"  Train games: {len(train_ids):_}, Val games: {len(val_ids):_}, Test games: {len(test_ids):_}")
train_df.to_csv(CONFIG.tetrio_train, index=False)
val_df.to_csv(CONFIG.tetrio_val, index=False)
test_df.to_csv(CONFIG.tetrio_test, index=False)

Loading processed dataset...
Other stuff...
Train games: 65_186, Val games: 3_834, Test games: 7_670
Saving partitions...


### Show dataloaders

In [ ]:
from src.data import load_tetrio_data

train_loader, test_loader, val_loader = load_tetrio_data(CONFIG, T_CONFIG)


In [ ]:
j = -1

In [ ]:
i = [7633,
9364,
9819,
9990,
12078,
13148,
14938,
21911,
23840,
24029,
28171,
28172,
34473,
34474,
38943,
49607,
52696,
53946,
55175,]
j += 1
print(val_loader.dataset.df.iloc[i[j]]['playfield'])
print(val_loader.dataset.df.iloc[i[j]]['playfield_next'])
print(val_loader.dataset.df.iloc[i[j]]['immediate_garbage'])
val_loader.dataset.df.iloc[i[j]]


```AssertionError at index 7633
AssertionError at index 9364
AssertionError at index 9819
AssertionError at index 9990
AssertionError at index 12078
AssertionError at index 13148
AssertionError at index 14938
AssertionError at index 21911
AssertionError at index 23840
AssertionError at index 24029
AssertionError at index 28171
AssertionError at index 28172
AssertionError at index 34473
AssertionError at index 34474
AssertionError at index 38943
AssertionError at index 49607
AssertionError at index 52696
AssertionError at index 53946
AssertionError at index 55175
```

In [ ]:
for i in range(len(val_loader.dataset)):
    try:
        data = val_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

In [ ]:
for i in range(len(test_loader.dataset)):
    try:
        data = test_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

In [ ]:
for i in range(len(train_loader.dataset)):
    try:
        data = train_loader.dataset[i]
    except AssertionError:
        print(f"AssertionError at index {i}")

In [ ]:
import sys, os
sys.path.insert(0, 'app/src')
sys.path.insert(0, 'app')
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pandas as pd
import numpy as np
from multiprocessing import Pool, cpu_count
import time

from src.config import Configuration
from src.tetris import TetrisConfiguration, Tetris, Board, MoveSearcher
from src.data.tetrio import find_board_index

from maikol_utils.print_utils import print_separator

CONFIG = Configuration()
T_CONFIG = TetrisConfiguration(vanish_zone=5)

def check_row(idx):
    row = _df.iloc[idx]
    g = int(row['immediate_garbage'])
    pf_rows = (len(row['playfield']) + 9) // 10
    game_h = max(20, pf_rows)
    
    game = Tetris(
        playfield=row['playfield'],
        next_pieces=row['next'],
        active_piece=row['placed'],
        hold_piece=row['hold'],
        vanish_zone=T_CONFIG.vanish_zone,
        height=game_h,
    )
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, feats = searcher.get_all_features()
    
    ts = row['playfield_next'][g*10:]
    board = Board(T_CONFIG.board_w, game_h, T_CONFIG.vanish_zone, False, ts)
    idx_match = find_board_index(board, feats['boards'])
    
    return idx, idx_match

def init_worker(df):
    global _df
    _df = df


def check(df):
    n = len(df)
    ncpu = int(cpu_count()*4/5)
    print(f"Rows: {n}, workers: {ncpu}")

    t0 = time.time()
    fails = []
    with Pool(processes=ncpu, initializer=init_worker, initargs=(df,)) as pool:
        for i, (idx, match) in enumerate(pool.imap_unordered(check_row, range(n), chunksize=100)):
            if match == -1:
                row = df.iloc[idx]
                fails.append((idx, row['game_id'], row['subframe'], row['placed']))
            if (i + 1) % 5000 == 0:
                elapsed = time.time() - t0
                rate = (i + 1) / elapsed
                print(f"  {i+1}/{n}  ({rate:.0f} rows/s)  fails: {len(fails)}")

    elapsed = time.time() - t0
    print(f"\nDone: {n} rows in {elapsed:.0f}s ({n/elapsed:.0f} rows/s)")
    print(f"Failures: {len(fails)}")

    if fails:
        for idx, gid, sf, placed in fails:
            print(f"  idx={idx} game={gid} frame={sf} placed={placed}")


df = pd.read_csv(CONFIG.tetrio_train)
print_separator("Checking TRAIN dataset")
check(df)
df = pd.read_csv(CONFIG.tetrio_test)
print_separator("Checking TEST dataset")
check(df)
df = pd.read_csv(CONFIG.tetrio_val)
print_separator("Checking VAL dataset")
check(df)


Train
```
Done: 5298785 rows in 2694s (1967 rows/s)
Failures: 36
  idx=152424 game=63692 frame=37 placed=I
  idx=206269 game=56528 frame=111 placed=T
  idx=459490 game=100868 frame=40 placed=I
  idx=992135 game=116213 frame=198 placed=O
  idx=1582055 game=41904 frame=6 placed=T
  idx=1654312 game=75388 frame=16 placed=O
  idx=1795276 game=87835 frame=21 placed=I
  idx=1999255 game=18764 frame=33 placed=I
  idx=2024786 game=41075 frame=25 placed=Z
  idx=2068541 game=76050 frame=188 placed=T
  idx=2168144 game=79993 frame=24 placed=I
  idx=2261471 game=54818 frame=33 placed=J
  idx=2283783 game=90662 frame=56 placed=I
  idx=2288823 game=96302 frame=100 placed=J
  idx=2295250 game=95381 frame=221 placed=I
  idx=2420530 game=118630 frame=22 placed=I
  idx=2430360 game=57202 frame=40 placed=S
  idx=2505352 game=28894 frame=19 placed=O
  idx=2883483 game=111087 frame=110 placed=I
  idx=3096109 game=17712 frame=51 placed=J
  idx=3122777 game=10834 frame=24 placed=T
  idx=3123617 game=31313 frame=56 placed=I
  idx=3153010 game=82466 frame=292 placed=J
  idx=3229186 game=111829 frame=13 placed=I
  idx=3320642 game=13540 frame=76 placed=I
  idx=3935599 game=33092 frame=33 placed=I
  idx=3963787 game=26120 frame=39 placed=I
  idx=4047455 game=21330 frame=178 placed=O
  idx=4147292 game=55830 frame=288 placed=Z
  idx=4310264 game=33190 frame=267 placed=I
  idx=4631116 game=25508 frame=100 placed=I
  idx=4641633 game=102480 frame=6 placed=Z
  idx=4732133 game=61688 frame=13 placed=I
  idx=4782245 game=39404 frame=81 placed=I
  idx=5056547 game=55137 frame=171 placed=T
  idx=5135853 game=12212 frame=276 placed=I
```
Test
```
Done: 1512354 rows in 751s (2014 rows/s)
Failures: 11
  idx=247619 game=23564 frame=29 placed=Z
  idx=284379 game=26990 frame=81 placed=L
  idx=334632 game=31230 frame=27 placed=T
  idx=603708 game=51982 frame=8 placed=Z
  idx=625684 game=53903 frame=3 placed=J
  idx=807995 game=69532 frame=129 placed=I
  idx=849085 game=73544 frame=12 placed=O
  idx=953869 game=82425 frame=32 placed=Z
  idx=1087022 game=93276 frame=29 placed=L
  idx=1364735 game=114115 frame=98 placed=O
  idx=1372526 game=114672 frame=27 placed=T
```

Val
```
Done: 752003 rows in 378s (1987 rows/s)
Failures: 6
  idx=104135 game=20506 frame=46 placed=I
  idx=211649 game=39588 frame=8 placed=L
  idx=232464 game=43028 frame=50 placed=I
  idx=239280 game=43796 frame=161 placed=S
  idx=268059 game=48600 frame=53 placed=I
  idx=709121 game=118296 frame=156 placed=L
```

6 + 11 + 36 = 53

# Show the data

In [66]:
game_recod = raw_df.sort_values(by=['game_id', 'subframe'])

frame = 0

In [79]:
from src.tetris import Tetris

playfield = game_recod.iloc[frame]['playfield']
playfield_next = game_recod.iloc[frame]['playfield_next']
next_pieces = game_recod.iloc[frame]['next']
current_piece = game_recod.iloc[frame]['real_current']
hold_piece = game_recod.iloc[frame]['real_hold']
# print(playfield)

game = Tetris(color_map=True, playfield=playfield, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state(include_vanish_zone=True)
frame += 1

..........
.....L....
...LLL....
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
..........
.ZZ.....JL
OOZZ..JJJL
Act.: L | Hold: I | Next: ['T', 'Z', 'S', 'O', 'I', 'L', 'S', 'Z', 'J', 'L', 'I', 'T', 'O', 'L']
Combo: ---    |  B2B: 0   |  Last Move: ---             | Total All Clears: 0    


In [ ]:
from src.tetris import MoveSearcher, Board, find_board_index

for i in range(1000):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()

import time 
start_time = time.time()
m = 10_000
for i in range(m):
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    _, features = searcher.get_all_features()
print(f"Time for {m} iterations: {time.time() - start_time:.2f} seconds, {(time.time() - start_time)/m:.4f} seconds per iteration")

In [ ]:

searcher = MoveSearcher(game, CONFIG, T_CONFIG)
_, features = searcher.get_all_features()
boards_batch = features['boards']   # (128, 24, 10)

board = Board(game.width, game.height, game.vanish_zone, game.color_map, playfield_next)
board.print_board(include_vanish_zone=True)
idx = find_board_index(board, boards_batch)
# idx is the matching index in features, or -1

idx

In [ ]:
game = Tetris(color_map=True, playfield=playfield_next, next_pieces=next_pieces, active_piece=current_piece, hold_piece=hold_piece)
game.print_state()

# Precompute features

In [ ]:
import glob
import os, time
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count
from src.tetris import Tetris, Board, MoveSearcher
from src.data.tetrio import find_board_index

_df = None
_out_dir = None

def init_worker(df):
    global _df
    _df = df

def precompute_row(idx):
    row = _df.iloc[idx]
    g = int(row['immediate_garbage'])
    pf_rows = (len(row['playfield']) + 9) // 10
    game_h = max(20, pf_rows)
    
    game = Tetris(
        playfield=row['playfield'],
        next_pieces=row['next'],
        active_piece=row['placed'],
        hold_piece=row['hold'],
        vanish_zone=T_CONFIG.vanish_zone,
        height=game_h,
    )
    searcher = MoveSearcher(game, CONFIG, T_CONFIG)
    game_state = [
        float(row['combo']),
        float(row['btb']),
        float(row['immediate_garbage']),
        float(row['incoming_garbage']),
    ]
    _, features = searcher.get_all_features(game_state)
    
    ts = row['playfield_next'][g*10:]
    board = Board(T_CONFIG.board_w, game_h, T_CONFIG.vanish_zone, False, ts)
    target = find_board_index(board, features['boards'])
    
    if target == -1:
        return idx, None
    
    bucket = idx // 1000
    bucket_dir = os.path.join(_out_dir, f"{bucket:04d}")
    os.makedirs(bucket_dir, exist_ok=True)
    path = os.path.join(bucket_dir, f"{idx:06d}.npz")
    np.savez_compressed(path,
        boards=features['boards'].astype(np.float32),
        queues=features['queues'].astype(np.float32),
        queue_idx=features['queue_idx'].astype(np.int64),
        game_state=features['game_state'],
        target=np.int64(target),
    )
    return idx, True

def precompute_split(df, out_dir, name, n_workers, csv_path=None):
    global _out_dir
    _out_dir = out_dir
    os.makedirs(out_dir, exist_ok=True)
    n = len(df)
    print(f"{name}: {n} rows, {n_workers} workers")
    
    t0 = time.time()
    fails = []
    valid = []
    with Pool(processes=n_workers, initializer=init_worker, initargs=(df,)) as pool:
        for i, (idx, ok) in enumerate(pool.imap_unordered(precompute_row, range(n), chunksize=50)):
            if ok is None:
                row = df.iloc[idx]
                fails.append((idx, row['game_id'], row['subframe']))
            else:
                valid.append(idx)
            if (i + 1) % 5000 == 0:
                elapsed = time.time() - t0
                eta = elapsed / (i + 1) * (n - i - 1)
                print(f"  {i+1}/{n}  (ETA {eta:.0f}s)  fails: {len(fails)}")
    
    # Save sorted valid indices for TetrioDataset
    valid.sort()
    with open(os.path.join(out_dir, 'valid_indices.txt'), 'w') as f:
        for idx in valid:
            f.write(f"{idx}\n")
    # Also save alongside CSV for TetrioDataset (live mode)
    if csv_path is not None:
        vi_path = csv_path.replace('.csv', '_valid.txt')
        with open(vi_path, 'w') as f:
            for idx in valid:
                f.write(f"{idx}\n")
        print(f'  Saved valid indices to {vi_path}')
    
    elapsed = time.time() - t0
    print(f"  Done: {n} rows in {elapsed:.0f}s ({n/elapsed:.0f} rows/s), {len(fails)} failures, {len(valid)} valid")
    if fails:
        print(f"  FAILURES: {len(fails)}")
        for idx, gid, sf in fails[:10]:
            print(f"    idx={idx} game={gid} frame={sf}")
    return len(fails)

# --- Run for all three partitions ---
n_workers = max(1, cpu_count() * 3 // 4)

for split_name, csv_path in [
    ('train', CONFIG.tetrio_train),
    ('val',   CONFIG.tetrio_val),
    ('test',  CONFIG.tetrio_test),
]:
    out_dir = os.path.join('data', 'precomputed', split_name)
    existing = glob.glob(os.path.join(out_dir, '*', '*.npz'))
    if len(existing) > 0:
        print(f'Skipping {split_name} — {len(existing)} files already present')
        continue
    df = pd.read_csv(csv_path)
    precompute_split(df, out_dir, split_name, n_workers, csv_path)

print('\nAll precomputed.')
